In [9]:
# Monkey-patch to get live output from python file into this notebook
import subprocess
import sys

_original_run = subprocess.run

def notebook_run(args, **kwargs):
    process = subprocess.Popen(
        [str(arg) for arg in args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()

    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, args)

    return subprocess.CompletedProcess(args, return_code)

subprocess.run = notebook_run


In [10]:
%load_ext autoreload
%autoreload 2

import importlib
import src.combine
importlib.reload(src.combine)

from src.combine import run_colmap
import open3d as o3d


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# run colmap
runit = run_colmap(image_dir="aloi_data/nerf_llff_data/fern/images", output_dir="outputs/demo")
# runit.run()   # this takes a long time to run, just use old outputs for testing changes to the run_colmap object

In [5]:
# visualize
ply_path = "outputs/demo/dense/fused.ply"

pcd = o3d.io.read_point_cloud(ply_path)

print(pcd)
print("Number of points:", len(pcd.points))

o3d.visualization.draw_geometries([pcd])

PointCloud with 1787638 points.
Number of points: 1787638


In [8]:
# remove noise
clean_pcd, noise_idx = runit.remove_noise()
# clean_pcd.paint_uniform_color([1, 0.706, 0])

# select noise
noise = pcd.select_by_index(noise_idx, invert=True)
noise.paint_uniform_color([1, 0, 0])

# visualize
o3d.visualization.draw_geometries([clean_pcd])

In [7]:
subsampled_pcd = runit.adaptive_subsample()
o3d.visualization.draw_geometries([clean_pcd])
print("Number of points:", len(subsampled_pcd.points))
print(f"Subsampling reduced number of points by {100 * (1 - len(subsampled_pcd.points) / len(pcd.points))}%")

Number of points: 774631
Subsampling reduced number of points by 56.66734540214517%


In [12]:
features = runit.extract_features()
features

{'fpfh': array([[ 9.03278673,  3.62812124,  2.40854089, ...,  3.69444866,
          2.75774334,  0.90436732],
        [19.21315702,  8.28944361, 33.79750812, ..., 26.02846055,
         31.82011873, 31.67827295],
        [17.8086003 , 15.26017538, 19.43185795, ..., 19.34039041,
         28.93673512, 22.97922404],
        ...,
        [ 6.83026591,  3.0189406 ,  5.96540449, ..., 22.00681365,
         27.69521055, 20.26243904],
        [18.27559175,  6.40699593,  1.22839501, ..., 17.6231185 ,
         22.61501262,  5.52109876],
        [11.93553493,  4.15435147,  1.90491839, ..., 14.67704961,
          3.01668458,  2.91573868]], shape=(1787638, 33)),
 'distance_to_centroid': array([14.91621089, 20.09083332, 19.98784966, ...,  8.14676736,
         9.3524227 , 11.43692696], shape=(1787638,)),
 'height': array([21.70418549, 30.57316971, 30.47636604, ..., 27.00764465,
        24.62170601, 25.68063164], shape=(1787638,))}